In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', None)


fulldata = pd.read_csv("/home/xutingfeng/infective_disease/data/Infective_disease_china-V3.csv")
fulldata = fulldata.dropna(subset=['Disease_en'])

data = fulldata[['Year/Month', 'Disease_en', "Case number"]].assign(City='global').pivot(
    index=['Year/Month','City'], columns='Disease_en', values='Case number'
).reset_index(drop=False, )
data

In [ ]:
from EpiAI.time_serie_task import train_prediction_model

model_configs = {
    # ===== Tabular 模型 =====
    "XGBSingle": {
        "modelType": "XGBSingle",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
        },
        "training_config": {
            "train_val_test_ratio": (6, 2, 2),
            "fillna_method": "zero",
            "normalize_x": True,
            "normalize_y": True,
            "use_val": True,
            "fit_verbose": False,
        },
    },
    # "LGBMSingle": {
    #     "modelType": "LGBMSingle",
    #     "params": {
    #         "lookback": 12,
    #         "horizon": 3,
    #         "batch_size": 32,
    #     },
    #     "training_config": {
    #         "train_val_test_ratio": (6, 2, 2),
    #         "fillna_method": "zero",
    #         "normalize_x": True,
    #         "normalize_y": True,
    #         "use_val": True,
    #         "fit_verbose": False,
    #     },
    # },
    "RandomForest": {
        "modelType": "RandomForest",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
        },
        "training_config": {
            "train_val_test_ratio": (6, 2, 2),
            "fillna_method": "zero",
            "normalize_x": True,
            "normalize_y": True,
            "use_val": True,
            "fit_verbose": False,
        },
    },
    "LinearReg": {
        "modelType": "LinearReg",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            # 可选：lr_params = {"fit_intercept": False}
        },
        "training_config": {
            "train_val_test_ratio": (6, 2, 2),
            "fillna_method": "zero",
            "normalize_x": True,
            "normalize_y": True,        # OLS 允许负值，归一化完全 OK
            "use_val": True,
            "fit_verbose": False,
        },
    },

    # "SVR": {
    #     "modelType": "SVR",
    #     "params": {
    #         "lookback": 12,
    #         "horizon": 3,
    #         "batch_size": 32,
    #         # 可选：自定义 SVM 参数（不写则用默认值 kernel="rbf", C=1.0）
    #         # "svm_params": {"C": 10.0, "gamma": "auto", "epsilon": 0.01},
    #     },
    #     "training_config": {
    #         "train_val_test_ratio": (6, 2, 2),
    #         "fillna_method": "zero",
    #         "normalize_x": True,
    #         "normalize_y": True,
    #         "use_val": True,
    #         "fit_verbose": False,
    #     },
    # },


    # ===== Torch 模型 =====
    "LSTM": {
        "modelType": "LSTM",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "hidden_dim": 128,
            "dropout": 0.1,
            "num_layers": 1,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "CNNLSTM": {
        "modelType": "CNNLSTM",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "cnn_channels": 32,
            "kernel_size": 3,
            "lstm_hidden": 64,
            "lstm_layers": 1,
            "dropout": 0.1,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "CNN": {
        "modelType": "CNN",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "dropout": 0.5,
            "linear_hid": 128,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "DLinear": {
        "modelType": "DLinear",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "kernel_size": 3,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "MLP": {
        "modelType": "MLP",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "hidden_dim": 128,
            "dropout": 0.1,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "ResNet": {
        "modelType": "ResNet",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "base_channels": 32,
            "num_blocks": 3,
            "kernel_size": 3,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "TCN": {
        "modelType": "TCN",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "channels": [32, 32],
            "kernel_size": 3,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "Transformer": {
        "modelType": "Transformer",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "d_model": 64,
            "nhead": 4,
            "num_layers": 2,
            "dim_feedforward": 128,
            "dropout": 0.1,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "Autoformer": {
        "modelType": "Autoformer",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "d_model": 128,
            "n_heads": 8,
            "e_layers": 2,
            "d_ff": 256,
            "moving_avg": 25,
            "factor": 1,
            "dropout": 0.1,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
    "TimesNet": {
        "modelType": "TimesNet",
        "params": {
            "lookback": 12,
            "horizon": 3,
            "batch_size": 32,
            "dropout": 0.1,
            "d_model": 64,
            "d_ff": 128,
            "e_layers": 2,
            "top_k": 5,
            "num_kernels": 6,
        },
        "training_config": {
            "max_epochs": 100,
            "lr": 1e-3,
            "weight_decay": 1e-6,
            "patience": 10,
            "train_val_test_ratio": (6, 2, 2),
            "save_best_path": None,
        },
    },
}


In [8]:
from EpiAI.time_serie_task import train_prediction_model
from copy import deepcopy
import os
import pickle
import json
from pathlib import Path

# params
time_col = 'Year/Month'

# 获取宽表中所有疾病列（排除非疾病列）
disease_columns = fulldata['Disease_en'].unique().tolist()

# 创建保存目录
savedir = Path("./simultation_platform_models")
savedir.mkdir(exist_ok=True, parents=True)

# 存储所有结果的字典
all_results = []

# 统计信息
total_tasks = len(disease_columns) * len(model_configs)
completed_tasks = 0
skipped_tasks = 0

for disease in disease_columns:
    for model_name, params in model_configs.items():
        
        print(f"\n{'='*60}")
        print(f"Processing: {disease} - {model_name}")
        print(f"Progress: {completed_tasks + skipped_tasks + 1}/{total_tasks}")
        print(f"{'='*60}")
        
        # 创建该疾病+模型的子目录
        model_savedir = savedir / disease.replace(" ", "_") / model_name
        model_savedir.mkdir(exist_ok=True, parents=True)
        
        # ========== 检查是否已存在训练结果 ==========
        required_files = [
            model_savedir / "predictions.csv",
            model_savedir / "metrics.csv",
            model_savedir / "model.pkl",
            model_savedir / "params.json"
        ]
        
        files_exist = all(f.exists() for f in required_files)
        
        if files_exist:
            print(f"✓ Skipping {disease} - {model_name} (already exists)")
            skipped_tasks += 1
            
            try:
                metric_df = pd.read_csv(model_savedir / "metrics.csv")
                result_summary = {
                    'Disease': disease,
                    'Model': model_name,
                    'MAE': metric_df.get('MAE', [None])[0],
                    'RMSE': metric_df.get('RMSE', [None])[0],
                    'R2': metric_df.get('R2', [None])[0],
                    'Savedir': str(model_savedir),
                    'Status': 'Skipped'
                }
                all_results.append(result_summary)
                print(f"  Loaded existing results.")
            except Exception as e:
                print(f"  Warning: Could not load existing results: {e}")
            
            continue
        
        # ========== 如果不存在，开始训练 ==========
        try:
            out = train_prediction_model(
                data=data,
                Case_col=disease,
                Feature_col=disease,
                Time_col=time_col,
                model_config=params,
            )
            
            result_df = out["result_df"]
            metrics_df = out["metrics_df"]
            model_obj = out["model"]
            
            # 1. 保存预测结果
            result_df.to_csv(
                model_savedir / "predictions.csv",
                index=False,
                encoding='utf-8-sig'
            )
            print(f"✓ Predictions saved to {model_savedir / 'predictions.csv'}")
            
            # 2. 保存评估指标
            metrics_df.to_csv(
                model_savedir / "metrics.csv",
                index=False,
                encoding='utf-8-sig'
            )
            print(f"✓ Metrics saved to {model_savedir / 'metrics.csv'}")
            
            # 3. 保存模型对象
            with open(model_savedir / "model.pkl", "wb") as f:
                pickle.dump(model_obj, f)
            print(f"✓ Model saved to {model_savedir / 'model.pkl'}")
            
            # 4. 保存模型参数
            with open(model_savedir / "params.json", "w", encoding='utf-8') as f:
                json.dump(params, f, indent=4, ensure_ascii=False)
            print(f"✓ Params saved to {model_savedir / 'params.json'}")
            
            # 5. 收集结果用于汇总
            metric_dict = metrics_df.iloc[0].to_dict() if len(metrics_df) > 0 else {}
            result_summary = {
                'Disease': disease,
                'Model': model_name,
                'MAE': metric_dict.get('MAE', None),
                'RMSE': metric_dict.get('RMSE', None),
                'R2': metric_dict.get('R2', None),
                'Savedir': str(model_savedir),
                'Status': 'Completed'
            }
            all_results.append(result_summary)
            
            completed_tasks += 1
            print(f"✓ {disease} - {model_name} completed successfully!")
            
        except Exception as e:
            print(f"✗ Error processing {disease} - {model_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
         
     
# ========== 保存汇总结果 ==========
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(savedir / "summary_results.csv", index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"All tasks completed!")
print(f"Total tasks: {total_tasks}")
print(f"Completed: {completed_tasks}")
print(f"Skipped: {skipped_tasks}")
print(f"Summary saved to {savedir / 'summary_results.csv'}")
print(f"{'='*60}\n")
print(summary_df)


Epoch [27/100] train_loss=0.909710 val_loss=0.778578 lr=1.000000e-03


 28%|██▊       | 28/100 [00:02<00:07,  9.19it/s]

Epoch [28/100] train_loss=0.909721 val_loss=0.778443 lr=1.000000e-03


 29%|██▉       | 29/100 [00:03<00:08,  8.62it/s]

Epoch [29/100] train_loss=0.909664 val_loss=0.778546 lr=1.000000e-03


...

 11%|█         | 11/100 [00:03<00:26,  3.30it/s]

Epoch [11/100] train_loss=0.275688 val_loss=0.000058 lr=1.000000e-03


 11%|█         | 11/100 [00:03<00:29,  2.98it/s]

Epoch [12/100] train_loss=0.276994 val_loss=0.000051 lr=1.000000e-03
Early stopping triggered at epoch 12.
Best val_loss: 0.000043


✓ Predictions saved to simultation_platform_models/Schistosomiasis/TimesNet/predictions.csv
✓ Metrics saved to simultation_platform_models/Schistosomiasis/TimesNet/metrics.csv
✓ Model saved to simultation_platform_models/Schistosomiasis/TimesNet/model.pkl
✓ Params saved to simultation_platform_models/Schistosomiasis/TimesNet/params.json
✓ Schistosomiasis - TimesNet completed successfully!

All tasks completed!
Total tasks: 533
Completed: 500
Skipped: 13
Summary saved to simultation_platform_models/summary_results.csv

             Disease         Model         MAE         RMSE        R2  \
0               AIDS     XGBSingle  405.459381   758.675691       NaN   
1               AIDS  RandomForest  546.414885   746.432363  0.651311   
2               AIDS     LinearReg  785.354062  1060.606948  0.296011   
3               AIDS          LSTM  892.568819  1172.059679  0.138951   
4               AIDS       CNNLSTM  917.007525  1185.407167  0.119228   
..               ...           ...    

In [ ]:
time_col = 'Year/Month'
disease = 'AIDS'
model_to_train = 'LGBMSingle'

out = train_prediction_model(
    data=data,
    Case_col=disease,
    Feature_col=disease,
    Time_col=time_col,
    model_config=model_configs[model_to_train],
)

trained_model = out["model"]
result_df = out["result_df"].rename(columns = {f"prediction_{disease}":model_to_train})
metrics_df = out["metrics_df"]
datamodule = out["datamodule"]


In [ ]:
metrics_df

In [ ]:
fig = plot_forecasting_results(
    data=final_df,
    x='Year/Month',
    y='Cases',
    hue='Source',
    title = target_feature_names[0]
)
